In [1]:
!pip install rasterio
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.5 MB/s eta 0:00:00


In [2]:
import gc
import json
import os
import time
from collections import defaultdict
from datetime import datetime

import cv2
import psutil
import rasterio
import numpy as np
import geopandas as gpd

from tqdm import tqdm
from shapely.ops import unary_union
from rasterio.windows import Window
from rasterio.transform import from_bounds
from shapely.geometry import Polygon, box as shapely_box

import matplotlib
matplotlib.use('Agg')
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


def _gpu_available():
    return TORCH_AVAILABLE and torch.cuda.is_available()


def setup_gpu_memory(vram_limit_gb=12.0):
    if not _gpu_available():
        print("  No GPU detected — running on CPU")
        return
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    frac     = min(vram_limit_gb / total_gb, 0.90)
    torch.cuda.set_per_process_memory_fraction(frac)
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {total_gb:.1f} GB total  →  {frac*100:.0f}% reserved "
          f"({frac*total_gb:.1f} GB)")


# ===========================================================================
# IMAGE READER
# ===========================================================================

class ImageReader:
    """
    Reads a basemap screenshot (PNG/JPEG/GeoTIFF) into memory.
    Stores the affine transform and CRS for downstream georeferencing.
    """

    def __init__(self, image_path: str):
        if not os.path.exists(image_path):
            raise FileNotFoundError(image_path)
        self.image_path = image_path
        self.data       = None          # (H, W, 3) uint8
        self.transform  = None
        self.crs        = None
        self.original_dims = None       # (width, height)
        self.band_count    = 3
        self._load()

    def _load(self):
        """Load the full image into RAM (basemap screenshots fit easily)."""
        try:
            with rasterio.open(self.image_path) as src:
                self.original_dims = (src.width, src.height)
                self.band_count    = src.count
                self.transform = src.transform
                self.crs       = src.crs
                if src.count >= 3:
                    data = src.read([1, 2, 3])
                    data = np.transpose(data, (1, 2, 0))
                else:
                    data = src.read(1)
                    data = cv2.cvtColor(data, cv2.COLOR_GRAY2RGB)
        except Exception:
            # Fallback: plain image (PNG/JPEG from basemap screenshot)
            import PIL.Image as PILImage
            img  = PILImage.open(self.image_path).convert('RGB')
            data = np.array(img)
            h, w = data.shape[:2]
            self.original_dims = (w, h)
            self.transform = from_bounds(0, 0, w, h, w, h)

        self.data = np.clip(data, 0, 255).astype(np.uint8)
        w, h = self.original_dims
        print(f"  Loaded {w}×{h} image  "
              f"({self.data.nbytes / 1024**2:.1f} MB in RAM)")


# ===========================================================================
# DIRECT (NON-TILED) YOLO-SEG INFERENCE
# ===========================================================================

def run_yolo_seg_direct(reader: ImageReader, model, config: dict) -> list[dict]:
    """
    Run YOLO11-seg directly on the full basemap screenshot — no tiling.

    The image is optionally downscaled to `infer_size` before being fed
    to the model; detected masks / polygons are then mapped back to the
    original pixel grid so that georeferencing works unchanged.
    """
    infer_size  = config.get('infer_size', 1280)   # px fed to YOLO
    conf        = config['conf_threshold']
    iou         = config['iou_threshold']
    max_det     = config.get('max_det', 300)
    device      = 'cuda' if _gpu_available() else 'cpu'
    class_names = model.names if hasattr(model, 'names') else {}

    orig_w, orig_h = reader.original_dims
    img_bgr = cv2.cvtColor(reader.data, cv2.COLOR_RGB2BGR)   # YOLO expects BGR

    # ── Optional resize to infer_size (keeps aspect ratio) ────────────
    scale = min(infer_size / orig_w, infer_size / orig_h, 1.0)
    infer_w = int(orig_w * scale)
    infer_h = int(orig_h * scale)

    if scale < 1.0:
        img_infer = cv2.resize(img_bgr, (infer_w, infer_h),
                               interpolation=cv2.INTER_LINEAR)
        print(f"  Resized {orig_w}×{orig_h} → {infer_w}×{infer_h} "
              f"(scale={scale:.4f}) for inference")
    else:
        img_infer = img_bgr
        print(f"  Inferring at full resolution: {orig_w}×{orig_h}")

    # ── YOLO inference ─────────────────────────────────────────────────
    print(f"  Running YOLO on single image  "
          f"(conf={conf}  iou={iou}  max_det={max_det})")
    results = model.predict(
        source=img_infer,
        conf=conf,
        iou=iou,
        max_det=max_det,
        verbose=False,
        device=device,
        retina_masks=True,
    )

    result = results[0]
    if result.masks is None or result.boxes is None:
        print("  No detections.")
        return []

    masks   = result.masks.data.cpu().numpy()    # (N, H_mask, W_mask)
    boxes   = result.boxes.xyxy.cpu().numpy()    # (N, 4) in infer-image space
    confs   = result.boxes.conf.cpu().numpy()
    cls_ids = result.boxes.cls.cpu().numpy().astype(int)

    all_detections = []

    for i in range(len(boxes)):
        # ── Mask → binary at original image resolution ─────────────────
        mask = masks[i]
        if mask.shape != (orig_h, orig_w):
            mask = cv2.resize(
                mask.astype(np.float32), (orig_w, orig_h),
                interpolation=cv2.INTER_LINEAR
            )
        binary = (mask > 0.5).astype(np.uint8)

        if binary.sum() < 9:
            continue

        # ── Contour → polygon ──────────────────────────────────────────
        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)
        if cv2.contourArea(contour) < 4:
            continue

        eps    = 0.002 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, eps, True)
        if len(approx) < 3:
            continue

        # Polygon in original-image pixel coords  (x, y)
        poly_px = approx.reshape(-1, 2).astype(np.int32)

        # ── BBox: scale from infer space → original-image space ────────
        bx1, by1, bx2, by2 = boxes[i]
        bbox_px = [
            int(bx1 / scale),
            int(by1 / scale),
            int(bx2 / scale),
            int(by2 / scale),
        ]

        cid = cls_ids[i]
        all_detections.append({
            'polygon_px': poly_px,
            'bbox_px':    bbox_px,
            'confidence': float(confs[i]),
            'class_id':   cid,
            'class_name': class_names.get(cid, str(cid)),
        })

    print(f"  ✓ {len(all_detections)} raw detections")
    return all_detections


# ===========================================================================
# POLYGON NMS
# ===========================================================================

def polygon_nms(detections: list[dict], iou_threshold: float = 0.4) -> list[dict]:

    if not detections:
        return detections

    det = sorted(detections, key=lambda d: d['confidence'], reverse=True)

    shapes = []
    for d in det:
        try:
            p = Polygon(d['polygon_px'])
            if not p.is_valid:
                p = p.buffer(0)
            shapes.append(p)
        except Exception:
            shapes.append(None)

    keep    = []
    removed = set()

    for i in range(len(det)):
        if i in removed or shapes[i] is None:
            continue
        keep.append(det[i])
        pi = shapes[i]
        ai = pi.area if pi else 0

        for j in range(i + 1, len(det)):
            if j in removed or shapes[j] is None:
                continue
            pj = shapes[j]
            try:
                inter = pi.intersection(pj).area
                if inter == 0:
                    continue
                union = ai + pj.area - inter
                if union > 0 and inter / union > iou_threshold:
                    removed.add(j)
            except Exception:
                continue

    print(f"  Polygon NMS: {len(det)} → {len(keep)}  (IoU>{iou_threshold})")
    return keep


# ===========================================================================
# GEOREFERENCING
# ===========================================================================

def detections_to_geodataframe(detections: list[dict],
                                transform,
                                crs) -> gpd.GeoDataFrame:
    """
    Convert pixel-space polygon detections to georeferenced GeoDataFrame.

    `transform` is the affine transform that maps image pixels to real-world
    coordinates — supplied either from the image's embedded geodata or from
    the basemap coordinate extraction helper (see config).
    """
    geo_polygons = []
    attrs        = []

    for d in tqdm(detections, desc="Georeferencing"):
        coords_px = d['polygon_px']   # (N, 2)  x, y
        try:
            geo_coords = [
                rasterio.transform.xy(transform, float(py), float(px))
                for px, py in coords_px
            ]
            geo_poly = Polygon(geo_coords)
            if not geo_poly.is_valid:
                geo_poly = geo_poly.buffer(0)
            if geo_poly.is_empty:
                continue
        except Exception:
            continue

        geo_polygons.append(geo_poly)
        attrs.append({
            'confidence':   d['confidence'],
            'class_id':     d['class_id'],
            'class_name':   d['class_name'],
            'area_m2':      geo_poly.area,
            'area_px':      Polygon(coords_px).area,
            'num_vertices': len(coords_px),
        })

    if not geo_polygons:
        return gpd.GeoDataFrame()

    gdf = gpd.GeoDataFrame(attrs, geometry=geo_polygons, crs=crs)
    print(f"  ✓ {len(gdf)} georeferenced polygons")
    return gdf


# ===========================================================================
# VISUALIZATION
# ===========================================================================

def visualize_segmentation(reader: ImageReader,
                            detections: list[dict],
                            config: dict) -> str:

    output_dir = config['output_dir']
    name       = config.get('image_name', 'output')
    max_dim    = config.get('max_viz_dimension', 4000)
    dpi        = config.get('viz_dpi', 150)

    orig_w, orig_h = reader.original_dims
    scale  = min(max_dim / orig_w, max_dim / orig_h, 1.0)
    out_w  = int(orig_w * scale)
    out_h  = int(orig_h * scale)

    print(f"  Canvas: {orig_w}×{orig_h} → {out_w}×{out_h}  scale={scale:.4f}")
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{name}_segmentation.png")

    # Background from the already-loaded full image
    bg = cv2.resize(reader.data, (out_w, out_h), interpolation=cv2.INTER_AREA)

    fig_w = out_w / dpi
    fig_h = out_h / dpi
    fig, ax = plt.subplots(figsize=(fig_w + 1.4, fig_h))

    ax.imshow(bg / 255.0, extent=[0, out_w, out_h, 0], aspect='equal', alpha=0.85)
    ax.set_xlim(0, out_w)
    ax.set_ylim(out_h, 0)
    ax.set_axis_off()
    ax.set_title("Crop Field Detections", fontsize=14, fontweight='bold', pad=10)

    cmap  = cm.plasma
    norm  = mcolors.Normalize(vmin=0.0, vmax=1.0)

    patch_list  = []
    conf_values = []

    for d in detections:
        pts = (d['polygon_px'] * scale).astype(np.float32)
        pts[:, 0] = np.clip(pts[:, 0], 0, out_w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, out_h - 1)
        if len(pts) < 3:
            continue
        patch_list.append(MplPolygon(pts, closed=True))
        conf_values.append(d['confidence'])

    if patch_list:
        face_colors = [cmap(norm(c)) for c in conf_values]
        fill_colors = [(r, g, b, 0.25) for r, g, b, _ in face_colors]
        pc = PatchCollection(
            patch_list,
            facecolors=fill_colors,
            edgecolors=[(1.0, 0.0, 0.0, 0.9)] * len(patch_list),
            linewidths=1.2,
        )
        ax.add_collection(pc)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Confidence', fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    n     = len(detections)
    avg_c = np.mean(conf_values) if conf_values else 0.0

    stats_lines = [f"Total: {n}", f"Avg Conf: {avg_c:.3f}"]
    class_counts: dict[str, int] = {}
    for d in detections:
        class_counts[d['class_name']] = class_counts.get(d['class_name'], 0) + 1
    if len(class_counts) > 1:
        for cname, cnt in sorted(class_counts.items()):
            stats_lines.append(f"{cname}: {cnt}")

    ax.text(
        0.02, 0.98, "\n".join(stats_lines),
        transform=ax.transAxes, fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  alpha=0.9, edgecolor="black", linewidth=0.8),
        fontfamily='monospace',
    )

    plt.tight_layout(pad=0.5)
    plt.savefig(out_path, dpi=dpi, bbox_inches='tight',
                pil_kwargs={'compress_level': 6})
    plt.close(fig)

    del bg, patch_list
    gc.collect()

    print(f"  ✓ PNG: {out_path}  ({os.path.getsize(out_path)/1024**2:.1f} MB)")
    return out_path


# ===========================================================================
# SAVE OUTPUTS
# ===========================================================================

def save_outputs(gdf: gpd.GeoDataFrame,
                 detections: list[dict],
                 config: dict,
                 elapsed_sec: float):
    output_dir = config['output_dir']
    name       = config.get('image_name', 'output')
    os.makedirs(output_dir, exist_ok=True)

    if gdf is not None and len(gdf) > 0 and gdf.crs is not None:
        p = os.path.join(output_dir, f"{name}_fields.geojson")
        try:
            gdf.to_file(p, driver='GeoJSON')
            print(f"  ✓ GeoJSON : {p}")
        except Exception as e:
            print(f"  ⚠ GeoJSON failed: {e}")

    if gdf is not None and len(gdf) > 0:
        p = os.path.join(output_dir, f"{name}_fields.csv")
        try:
            gdf.drop(columns='geometry').to_csv(p, index=False)
            print(f"  ✓ CSV     : {p}")
        except Exception as e:
            print(f"  ⚠ CSV failed: {e}")

    if detections:
        confs = [d['confidence'] for d in detections]
        areas = [Polygon(d['polygon_px']).area for d in detections
                 if len(d['polygon_px']) >= 3]
        class_counts: dict[str, int] = {}
        for d in detections:
            class_counts[d['class_name']] = class_counts.get(d['class_name'], 0) + 1

        stats = {
            'total_fields':        len(detections),
            'class_counts':        class_counts,
            'avg_confidence':      float(np.mean(confs)),
            'min_confidence':      float(np.min(confs)),
            'max_confidence':      float(np.max(confs)),
            'avg_area_px':         float(np.mean(areas)) if areas else 0,
            'median_area_px':      float(np.median(areas)) if areas else 0,
            'processing_time_min': round(elapsed_sec / 60, 2),
        }
        p = os.path.join(output_dir, f"{name}_statistics.json")
        try:
            with open(p, 'w') as f:
                json.dump(stats, f, indent=2)
            print(f"  ✓ Stats   : {p}")
        except Exception as e:
            print(f"  ⚠ Stats failed: {e}")


# ===========================================================================
# MAIN PIPELINE
# ===========================================================================

def run_cropfield_segmentation(image_path: str,
                                model,
                                config: dict):
    """
    Direct-inference pipeline for basemap screenshots:
      1. Load full screenshot into RAM
      2. Single-pass YOLO11-seg inference (no tiling)
      3. Polygon NMS
      4. Georeferencing → GeoDataFrame  (using basemap-extracted transform)
      5. Visualization
      6. Save GeoJSON, CSV, statistics

    """
    t0      = time.time()
    process = psutil.Process()

    def _ram():
        return process.memory_info().rss / 1024**3

    print(f"\n{'='*65}")
    print("  CROP FIELD SEGMENTATION — DIRECT INFERENCE")
    print(f"{'='*65}")
    print(f"  Image      : {image_path}")
    print(f"  Model      : {config.get('model_path', 'provided externally')}")
    print(f"  Output dir : {config['output_dir']}")

    # ── Load image ─────────────────────────────────────────────────
    print(f"\n[1/5] Loading basemap screenshot…")
    reader = ImageReader(image_path)
    orig_w, orig_h = reader.original_dims
    print(f"  ✓ {orig_w}×{orig_h}  CRS={reader.crs}  RAM={_ram():.2f} GB")

    # ── Direct YOLO-seg ────────────────────────────────────────────
    print(f"\n[2/5] Running YOLO11-seg (direct, no tiling)…")
    detections = run_yolo_seg_direct(reader, model, config)
    print
    (f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    if not detections:
        print("  ⚠ No fields detected. Check conf_threshold and model path.")
        return None, []

    # ── Polygon NMS ────────────────────────────────────────────────
    print(f"\n[3/5] Polygon NMS…")
    nms_iou    = config.get('polygon_nms_iou', 0.4)
    detections = polygon_nms(detections, nms_iou)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    # ── Georeference ───────────────────────────────────────────────
    print(f"\n[4/5] Georeferencing…")
    gdf = detections_to_geodataframe(detections, reader.transform, reader.crs)
    print(f"  RAM: {_ram():.2f} GB  |  t={time.time()-t0:.1f}s")

    # ── Visualization ──────────────────────────────────────────────
    print(f"\n[5/5] Visualization…")
    visualize_segmentation(reader, detections, config)

    # ── Save ──────────────────────────────────────────────────────────
    elapsed = time.time() - t0
    print(f"\n[Save] Writing outputs…")
    save_outputs(gdf, detections, config, elapsed)

    print(f"\n{'='*65}")
    print(f"  ✓ Done in {elapsed/60:.1f} min  |  {len(detections)} fields")
    print(f"  Peak RAM: {_ram():.2f} GB")
    print(f"{'='*65}\n")

    return gdf, detections

In [6]:
if __name__ == "__main__":
    import torch
    from ultralytics import YOLO

    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('medium')

    setup_gpu_memory(vram_limit_gb=12.0)

    # ── Load model ────────────────────────────────────────────────────
    MODEL_PATH = (
        '/content/drive/MyDrive/AGRI/CropField_Segmentation/result/'
        'yolov11n-seg-cropfield-100epoch-version03/weights/best.pt'
    )
    model = YOLO(MODEL_PATH)
    model.fuse()
    if torch.cuda.is_available():
        model.half()   # FP16 — faster on T4, lower VRAM

    # ── Config ────────────────────────────────────────────────────────
    config = {
        # Inference
        'infer_size':       640,   # longest side (px) fed to YOLO;

        # Detection thresholds
        'conf_threshold':   0.10,
        'iou_threshold':    0.30,   # YOLO internal NMS
        'max_det':          1000,
        'polygon_nms_iou':  0.20,

        # Visualization
        'max_viz_dimension': 4000,
        'viz_dpi':           150,

        # Output
        'output_dir':  '/content/drive/MyDrive/AGRI/CropField_Segmentation/output',
        'image_name':  'cropfield_seg',
        'model_path':  MODEL_PATH,
    }

    IMAGE_PATH = (
        '/content/drive/MyDrive/AGRI/CropField_Segmentation/'
        'Screenshot 2026-04-10 131755.png'
    )

    gdf, detections = run_cropfield_segmentation(IMAGE_PATH, model, config)


  GPU  : Tesla T4
  VRAM : 14.6 GB total  →  82% reserved (12.0 GB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs

  CROP FIELD SEGMENTATION — DIRECT INFERENCE
  Image      : /content/drive/MyDrive/AGRI/CropField_Segmentation/Screenshot 2026-04-10 131755.png
  Model      : /content/drive/MyDrive/AGRI/CropField_Segmentation/result/yolov11n-seg-cropfield-100epoch-version03/weights/best.pt
  Output dir : /content/drive/MyDrive/AGRI/CropField_Segmentation/output

[1/5] Loading basemap screenshot…
  Loaded 895×636 image  (1.6 MB in RAM)
  ✓ 895×636  CRS=None  RAM=1.70 GB

[2/5] Running YOLO11-seg (direct, no tiling)…
  Resized 895×636 → 640×454 (scale=0.7151) for inference
  Running YOLO on single image  (conf=0.1  iou=0.3  max_det=1000)


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


  ✓ 87 raw detections
  RAM: 1.70 GB  |  t=0.2s

[3/5] Polygon NMS…
  Polygon NMS: 87 → 86  (IoU>0.2)
  RAM: 1.70 GB  |  t=0.3s

[4/5] Georeferencing…


Georeferencing: 100%|██████████| 86/86 [00:00<00:00, 493.02it/s]

  ✓ 86 georeferenced polygons
  RAM: 1.70 GB  |  t=0.5s

[5/5] Visualization…
  Canvas: 895×636 → 895×636  scale=1.0000


  ✓ PNG: /content/drive/MyDrive/AGRI/CropField_Segmentation/output/cropfield_seg_segmentation.png  (0.8 MB)

[Save] Writing outputs…
  ✓ CSV     : /content/drive/MyDrive/AGRI/CropField_Segmentation/output/cropfield_seg_fields.csv
  ✓ Stats   : /content/drive/MyDrive/AGRI/CropField_Segmentation/output/cropfield_seg_statistics.json

  ✓ Done in 0.0 min  |  86 fields
  Peak RAM: 1.65 GB

